# Mamba-CL under the Inf-SSM protocol on Split-CIFAR100 (10 tasks x 10 classes): Vim-small backbone

Our first Mamba-CL run (`2026-10-19_mamba_cl_base_backbone.ipynb`, AA 79.52 / FM 8.53) used Mamba-CL's own protocol
(separate head per task, Adam lr 2e-4, AutoAugment, last epoch), so it could not be compared with the Inf-SSM run
(AA 28.42 / AIA 55.35 / FM 24.83). This notebook runs **Mamba-CL's method under exactly the Inf-SSM protocol**: it is
the Inf-SSM trainer in `baselines/inf-ssm` with `--cl_method mamba_cl` (`mamba_cl.py`).

**From Mamba-CL (the method):** null-space projection of every update of `x_proj_{step,B,C}`, `A_log` / `A_b_log`
(both scan directions) and `out_proj.weight` of each block (216 parameters), Mamba-CL's adaptive SVD threshold,
covariances summed over all tasks so far, `NULL_ETA` = 0.95 (the paper's CIFAR-100 value). As in Mamba-CL only these
parameters and the head are trained; the rest of the backbone is frozen. With `OPTIM = "mamba_cl"` (default) they
are also trained with **Mamba-CL's optimisation**: Adam lr 2e-4, head LR x50, weight decay 5e-5 (none on the head),
no per-task LR scaling, LR x0.1 at epochs 20 and 32. The head has no weight decay because, with one shared head, the
old-class rows get no gradient and Adam's L2 term alone erased them (run `cce192e0ecad`: old tasks 0%, AA 9.88). The Inf-SSM LR (1e-6 falling to 1e-7) is made for training all 26M weights;
with only these parameters it learns nothing after task 1 (our first run: T1 97%, every later task ~0%).

**From the Inf-SSM protocol (same as the Inf-SSM and sequential fine-tuning runs):** Vim-small from the authors'
task-0 checkpoint, natural class order, shared 100-way head with logits restricted to the current task in training,
10 tasks, 40 epochs per task, batch 128, mixup/cutmix, fp32,
best-val epoch per task, class-incremental evaluation (AA / AIA / FM).

Environment is the same as the Inf-SSM notebook (Python 3.11 venv, torch 2.3.1 + cu121, causal-conv1d 1.4.0,
mamba-ssm 1.2.2), plus `scipy` for the SVD threshold.

**Settings:** GPU **T4 x2**, Internet on. One run uses both GPUs (`DATA_PARALLEL`): each batch of 128 is split 64 + 64.
The null-space update after each task (collect + 216 SVDs) runs on one GPU.
`task_end.pt` also holds the summed covariances (~0.5 GB), so each task upload to `HF_REPO` is larger than in the Inf-SSM run.
Attach these Kaggle Secrets (Add-ons > Secrets):
- `HF_TOKEN`: a Hugging Face token with **write** access. Checkpoints and `results.json` are mirrored to the private
  repo `HF_REPO`, so they survive the end of the session.
- `WANDB_API_KEY` (optional). Without it W&B logs offline; sync later with `wandb sync` on the saved `wandb/` folder.

**Resuming:** Kaggle stops sessions at 12 h. Start a new session and **Run All** with the same parameters. The run
pulls its checkpoints from `HF_REPO` and continues: from the last finished task (`task_end.pt`, `results.json`), or
inside a task from the last uploaded epoch (`epoch.pt` + `task_best.pt`, every `HF_SYNC_EVERY` epochs).

**First run on a new setup:** do a smoke run with `TASKS = 10`, `EPOCHS = 1`. The log should show
`:: mamba_cl: 216 null-space parameters`, a `null-space update took ...` line after task 1, and from task 2 on
`Mamba-CL null-space projection eta=0.95 on 216 parameters`.

In [ ]:
# ---- run parameters ----
REPO     = "https://github.com/HasinthakaPiyumal/hippo-cortex.git"
BRANCH   = "feat/inf-ssm-vim"
DATASET  = "cifar100"   # cifar100 | imagenet_r | caltech256
TASKS    = 10           # 5 or 10 (the paper hyperparameters for that split are picked automatically)
SEED     = 0            # paper: 0, 10, 100. Only seed 0 has a released task-0 checkpoint
EPOCHS   = 40           # paper
BATCH    = 128          # paper; with DATA_PARALLEL each T4 gets 64
CL_METHOD = "mamba_cl"  # mamba_cl = Mamba-CL null-space projection (this notebook); reg = Inf-SSM; seq = no CL method
NULL_ETA = 0.95         # Mamba-CL: 0.95 (0.90 for 10-split ImageNet-R)
OPTIM = "mamba_cl"      # mamba_cl = Mamba-CL's Adam 2e-4 / head x50 / multistep; inf_ssm = the Inf-SSM AdamW 1e-5 (learns nothing here)
FULL_RUN = SEED != 0    # also train task 0, from ImageNet Vim-small (needed for seeds without a released checkpoint)
FINETUNE = "vim_small_in1k" if FULL_RUN else "hf"
USE_AMP  = False        # paper runs fp32; True is faster on T4 but changes the numerics
DATA_PARALLEL = True    # split each batch across both T4s (one run); False = single GPU
GRAD_CKPT = False       # per-block activation recomputation: ~6.5 -> 1.2 GB per T4 at 32 images, ~10% slower. True if memory runs out
MICRO_BATCH = 64        # run each batch of 128 in chunks of 64 (32 per T4) with gradient accumulation: same update, less memory
RUN_PROBE = False       # before training, print peak memory and speed per batch size on one T4 (about 2 min)
WORKERS  = 4
WANDB_PROJECT = "mamba-cl-baseline"
PROGRESS_INTERVAL = 60  # seconds between progress-bar updates in the log

# ---- checkpoints ----
CKPT_DIR = "/kaggle/working/checkpoints"
HF_REPO  = "hasinthakapiyumal/hippocortex-checkpoints"   # private model repo, created on first run
HF_SYNC_EVERY = 5        # upload the mid-task checkpoint every N epochs (~0.4 GB each); 0 = only at task ends
HF_SQUASH = True         # squash HF_REPO's history after each task, so it keeps one copy per file (use a dedicated repo)

In [ ]:
!nvidia-smi
import os, subprocess
os.chdir("/kaggle/working")
if not os.path.exists("hippo-cortex"):
    # Fall back to main once the branch is merged (GitHub deletes merged branches here).
    if subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO]).returncode != 0:
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
ROOT = "/kaggle/working/hippo-cortex"
VENV = "/kaggle/working/py311_env"
PY, PIP = f"{VENV}/bin/python", f"{VENV}/bin/pip"
!cd {ROOT} && git log --oneline -1

In [ ]:
# Python 3.11 venv (the Kaggle image's Python is too new for torch 2.3.1 / mamba-ssm 1.2.2)
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null
!apt-get update -y -qq > /dev/null
!apt-get install -y -qq python3.11 python3.11-venv python3.11-dev python3.11-distutils > /dev/null
!python3.11 -m venv {VENV}
!{PY} -m ensurepip
!{PIP} install -q --upgrade pip
!{PIP} install -q torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121
!{PIP} install -q ninja packaging wheel
!{PIP} install -q --no-build-isolation causal-conv1d==1.4.0
!{PIP} install -q --no-build-isolation mamba-ssm==1.2.2
# transformers must stay at 4.41.2 (as in requirements.txt): mamba-ssm 1.2.2 imports it, and newer releases
# refuse torch 2.3.1, which breaks the mamba import.
!{PIP} install -q numpy==1.26.4 timm==1.0.9 einops==0.8.0 transformers==4.41.2 gdown tqdm huggingface_hub scipy && {PIP} install -q -U wandb
# The model must find the CUDA kernels; without them it falls back to the (very slow) PyTorch reference scan.
# check=True stops Run All here if the kernels are missing (a failing "!" command would not).
subprocess.run([PY, "-c", "import torch, mamba_vim as m; assert not m.KERNEL_IMPORT_ERRORS, m.KERNEL_IMPORT_ERRORS; "
                "print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), torch.cuda.device_count(), 'GPU(s) | mamba + causal-conv1d kernels OK')"],
               cwd=f"{ROOT}/baselines/inf-ssm", check=True)

In [ ]:
# Data. CIFAR-100 comes from the team's Google Drive copy (scripts/download_datasets.py, as in Mamba-CL);
# ImageNet-R / Caltech-256 are image folders.
DATA = {"cifar100": "/kaggle/working/data/cifar100",
        "imagenet_r": "/kaggle/working/data/imagenet-r",
        "caltech256": "/kaggle/working/data/256_ObjectCategories"}[DATASET]
os.makedirs("/kaggle/working/data", exist_ok=True)
if DATASET == "imagenet_r" and not os.path.exists(DATA):
    !wget -q -O /tmp/imagenet-r.tar https://people.eecs.berkeley.edu/~hendrycks/imagenet-r.tar && tar -xf /tmp/imagenet-r.tar -C /kaggle/working/data && rm /tmp/imagenet-r.tar
if DATASET == "caltech256" and not os.path.exists(DATA):
    !wget -q -O /tmp/caltech256.tar https://data.caltech.edu/records/nyy15-4j048/files/256_ObjectCategories.tar && tar -xf /tmp/caltech256.tar -C /kaggle/working/data && rm /tmp/caltech256.tar
if DATASET == "cifar100":
    !cd {ROOT} && {PY} scripts/download_datasets.py --root /kaggle/working/data --dataset cifar100
!ls {DATA} | head -3

In [ ]:
# Tokens come from Kaggle Secrets; they are never written into this notebook.
env = dict(os.environ, WANDB_PROJECT=WANDB_PROJECT, WANDB_DIR="/kaggle/working")
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
env["HF_TOKEN"] = secrets.get_secret("HF_TOKEN").strip()   # required: checkpoints are mirrored to HF_REPO
# Check the token here, before training starts, rather than failing inside train_eval.py.
from huggingface_hub import HfApi
me = HfApi(token=env["HF_TOKEN"]).whoami()   # 401 here = the HF_TOKEN secret is wrong, expired or revoked
spaces = [me["name"]] + [o["name"] for o in me.get("orgs", [])]
assert HF_REPO.split("/")[0] in spaces, f"HF_REPO namespace must be one of {spaces}"
print(f"HF: logged in as {me['name']}, mirroring checkpoints to {HF_REPO}")
try:
    env["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
    print("W&B: online (key from Kaggle Secret)")
except Exception as e:
    env["WANDB_MODE"] = "offline"
    print("W&B: secret WANDB_API_KEY not attached, logging offline:", type(e).__name__)

In [ ]:
# Peak memory and step time of one Inf-SSM step on one T4, per batch size: use it to set MICRO_BATCH / GRAD_CKPT.
# With DATA_PARALLEL each T4 sees MICRO_BATCH / 2 images.
if RUN_PROBE:
    !cd {ROOT}/baselines/inf-ssm && {PY} mem_probe.py --batches 8 16 32 64 --ckpt true false --amp {str(USE_AMP).lower()}

In [ ]:
import time
LOG = f"/kaggle/working/inf_ssm_{DATASET}_t{TASKS}_seed_{SEED}_{time.strftime('%Y%m%d-%H%M%S')}.log"   # one log per session
cmd = [PY, "-u", "train_eval.py", "-d", DATASET, "-dr", DATA, "-t", str(TASKS),
       "-e", str(EPOCHS), "-b", str(BATCH), "--seed", str(SEED),
       "--cl_method", CL_METHOD, "--finetune", FINETUNE, *(["--full_run"] if FULL_RUN else []),
       "--use_amp", str(USE_AMP), "--data_parallel", str(DATA_PARALLEL), "--grad_ckpt", str(GRAD_CKPT), "--micro_batch", str(MICRO_BATCH),
       "--null_eta", str(NULL_ETA), "--optim", OPTIM,
       "-jt", str(WORKERS), "--use_wandb", "--wandb_project", WANDB_PROJECT,
       "--progress_interval", str(PROGRESS_INTERVAL),
       "--ckpt_dir", CKPT_DIR, "--hf_repo", HF_REPO,
       "--hf_sync_every", str(HF_SYNC_EVERY), "--hf_squash", str(HF_SQUASH)]
print(" ".join(cmd))
RUN_NAME = None
t0 = time.time(); done = 0
with open(LOG, "w") as f:
    p = subprocess.Popen(cmd, cwd=f"{ROOT}/baselines/inf-ssm", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        f.write(line); f.flush()
        print(line, end="", flush=True)   # full log: epochs, val accuracy, per-task results, checkpoints
        if line.startswith(":: run "):
            RUN_NAME = line.split()[2]      # the checkpoint folder / HF subfolder name
        if "Results of task" in line:
            done += 1
            el = time.time() - t0
            print(f"   this session: {done} task(s) in {el/3600:.2f} h, {el/done/3600:.2f} h per task", flush=True)
    p.wait()
print("exit code", p.returncode, f"in {(time.time()-t0)/3600:.2f} h; full log: {LOG}")

In [ ]:
# Keep this session's log next to the checkpoints on HF, then show the results so far.
import json
from huggingface_hub import HfApi
if RUN_NAME:
    try:
        HfApi(token=env["HF_TOKEN"]).upload_file(path_or_fileobj=LOG, path_in_repo=f"{RUN_NAME}/logs/{os.path.basename(LOG)}",
                                                 repo_id=HF_REPO, commit_message="session log")
        print("log uploaded to", f"{HF_REPO}/{RUN_NAME}/logs/")
    except Exception as e:
        print("log upload failed:", e)

res_path = f"{CKPT_DIR}/{RUN_NAME}/results.json"
if RUN_NAME and os.path.exists(res_path):
    r = json.load(open(res_path))
    print(f"tasks done {r['tasks_done']}/{r['num_tasks']} | AA {r['avg_acc']:.2%} | AIA {r['avg_inc_acc']:.2%} | FM {r['forgetting']:.2%} | {r['elapsed_hours']:.2f} h")
    print("AA per task:", " ".join(f"{a:.2%}" for a in r["avg_acc_per_task"]))
    print("same protocol, seed 0: Inf-SSM AA 28.42% | AIA 55.35% | FM 24.83%")
else:
    print("no results.json yet")

In [ ]:
!tail -n 40 {LOG}